# PerGAZE GazeformerISP on Google Colab: one GPU

Select **Runtime > Change runtime type > GPU**, then run cells in order.
Mount Google Drive and edit the paths below. The notebook uses only cuda:0.

Train: train.json. Validation: test_seen.json. Defaults: seed 10, 5 supervised
and 5 RL epochs, one warmup epoch, max_length 16, hidden dimension 512 and
6 encoder/6 decoder layers. Images/JSON are read directly from Drive.
The frozen COCO ResNet50 processes each batch online: no image feature cache.
SentenceTransformer stsb-roberta-base-v2 produces 768D text vectors once.

TP/TA subject IDs belong to different populations. TA text describes the search
instruction; TP text names the object; VQA text is the question. Scanpath frame
is 512x352, with 24x32 spatial actions plus STOP. Checkpoints, text NPZ and
reports are saved to Drive for resume and later GazeformerISP-S adaptation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import subprocess
import sys
from pathlib import Path

CODE = Path("/content/IndividualScanpath")
if (CODE / ".git").is_dir():
    subprocess.run(["git", "-C", str(CODE), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ntnghia06/IndividualScanpath.git", str(CODE)], check=True)
SRC = CODE / "PerGAZE/GazeformerISP/src"
assert (SRC / "train.py").is_file()


In [ ]:
%pip install -q "numpy>=1.24" "Pillow>=9.1" "scipy>=1.10" "tqdm>=4.65" multimatch-gaze sentence-transformers


In [ ]:
import torch
import torchvision

assert torch.cuda.is_available(), "Select GPU in Runtime > Change runtime type"
GPU_IDS = [0]
print("PyTorch:", torch.__version__, "torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)


## Configure Drive paths and training

Example layout (matching the GazeformerISP-S Colab notebook):

```text
MyDrive/PerGAZE/
  dataset/
    train.json
    test_seen.json
    support.json         # Optional now; include for later ISP-S.
    test_unseen.json     # Optional now; include for later ISP-S.
    images/
  pergaze_gazeformer_run/ # New training output.
  gazeformer_task_embeddings.npz
```

Edit DRIVE_ROOT, DATA, RUN and TEXT_FILE if needed. Inputs are read directly
from Drive, without copying images or JSON to /content. Attention maps are not
used by Gazeformer. Local storage holds code, temporary logs and optional ZIPs.

Batch defaults are 1 on a single GPU. To resume, set RESUME=True, point RUN to
the existing full run, and retain the exact original text NPZ and JSON files.
Keep training batch, seed, max_length and RL schedule consistent with the saved
configuration. A total EPOCHS target may be extended; it is not extra epochs.


In [ ]:
SEED = 10
EPOCHS = 10
START_RL_EPOCH = 5  # Zero-based: supervised epochs 1-5, RL epochs 6-10.
MAX_LENGTH = 16
TRAIN_BATCH = 1
TEST_BATCH = 1
WORKERS = 2
RESUME = False
VALIDATE_DATA = True
RUN_SMOKE = True

DRIVE_ROOT = Path("/content/drive/MyDrive/PerGAZE")
DATA = DRIVE_ROOT / "dataset"
RUN = DRIVE_ROOT / "pergaze_gazeformer_run"
TEXT_FILE = DRIVE_ROOT / "gazeformer_task_embeddings.npz"
LOCAL_ROOT = Path("/content/pergaze_gazeformer_work")

for name in ("train.json", "test_seen.json", "images"):
    assert (DATA / name).exists(), f"Missing: {DATA / name}"
assert min(EPOCHS, MAX_LENGTH, TRAIN_BATCH, TEST_BATCH) >= 1
assert START_RL_EPOCH >= 0 and WORKERS >= 0
RUN.mkdir(parents=True, exist_ok=True)
if RESUME:
    assert (RUN / "checkpoints/checkpoint.pth").is_file(), f"No resume checkpoint at {RUN}"
    assert TEXT_FILE.is_file(), f"Keep the original text NPZ for resume: {TEXT_FILE}"
print("Dataset on Drive:", DATA)
print("Persistent run output:", RUN)
print("Persistent text vectors:", TEXT_FILE)


In [ ]:
import shutil

COMMON = [
    "--train_file", str(DATA / "train.json"),
    "--val_file", str(DATA / "test_seen.json"),
    "--img_dir", str(DATA / "images"),
    "--text_embeddings", str(TEXT_FILE),
    "--seed", str(SEED), "--max_length", str(MAX_LENGTH),
    "--device", "cuda:0", "--gpu_ids", "0",
]

def run_script(script, *arguments, include_common=True):
    import codecs
    import time

    command = [sys.executable, "-u", str(SRC / script), *(COMMON if include_common else []), *arguments]
    local_logs = LOCAL_ROOT / "logs"
    local_logs.mkdir(parents=True, exist_ok=True)
    log_path = local_logs / f"{Path(script).stem}_{time.time_ns()}.log"
    drive_log = RUN / "logs" / log_path.name
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    tail = bytearray()
    process = None
    return_code = None
    print("Full subprocess log:", log_path, flush=True)
    try:
        with log_path.open("wb") as log:
            log.write(("Command: " + repr(command) + "\n").encode("utf-8"))
            process = subprocess.Popen(command, stdout=subprocess.PIPE,
                                       stderr=subprocess.STDOUT)
            try:
                while True:
                    chunk = process.stdout.read1(8192)
                    if not chunk:
                        break
                    log.write(chunk)
                    log.flush()
                    tail.extend(chunk)
                    if len(tail) > 65536:
                        del tail[:-65536]
                    sys.stdout.write(decoder.decode(chunk))
                    sys.stdout.flush()
                sys.stdout.write(decoder.decode(b"", final=True))
                return_code = process.wait()
            finally:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                process.stdout.close()
    finally:
        if log_path.is_file():
            try:
                drive_log.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(log_path, drive_log)
                print("Log copied to Drive:", drive_log, flush=True)
            except OSError as error:
                print("Could not copy log to Drive; local log is retained:", error, flush=True)
    if return_code:
        lines = tail.decode("utf-8", errors="replace").replace("\r", "\n").splitlines()
        last_lines = "\n".join(lines[-40:])
        hint = ""
        if "out of memory" in last_lines.lower():
            hint = "\nMemory error: for a NEW run try TRAIN_BATCH=1, TEST_BATCH=1. Resume requires the saved train batch."
        elif "Existing checkpoint" in last_lines:
            hint = "\nSet RESUME=True to continue that run, or choose a new RUN directory."
        raise RuntimeError(f"{script} exited with code {return_code}. Full log: {log_path}"
                           f"\nLast subprocess output:\n{last_lines}{hint}")


## Prepare pretrained sentence embeddings

This step downloads stsb-roberta-base-v2 if needed and encodes unique task text
from train/test_seen and, when present, support/test_unseen. It uses only text,
not scanpath labels. Including unseen questions here allows the original NPZ to
be reused later by ISP-S. Only text vectors are saved; image features are not.

A complete existing NPZ is reused. On RESUME the original NPZ is preserved
without regeneration; the train script checks its checkpoint SHA-256. For a
new dataset/text inventory, use a different TEXT_FILE and new RUN. Do not
replace an old run's text archive while resuming.


In [ ]:
if RESUME:
    print("Reusing original checkpoint text NPZ:", TEXT_FILE)
else:
    TEXT_FILES = [DATA / name for name in
                  ("train.json", "test_seen.json", "support.json", "test_unseen.json")
                  if (DATA / name).is_file()]
    run_script("preprocess/feature_extractor.py", "--mode", "text",
               "--files", *map(str, TEXT_FILES), "--output", str(TEXT_FILE),
               "--device", "cuda:0", include_common=False)
assert TEXT_FILE.is_file()


## Validate data and optional smoke check

Validation checks the train/validation records and original images. The smoke
check uses a small random transformer and random backbone to verify
forward/backward/inference/RL. Main training restores the full default architecture
and pretrained COCO backbone. Smoke output is local and is not a trained run.


In [ ]:
if VALIDATE_DATA:
    run_script("preprocess/validate_dataset.py", "--log_root", str(RUN))
if RUN_SMOKE and not RESUME:
    run_script("train.py", "--smoke_test", "--batch", "1", "--max_length", "3",
               "--hidden_dim", "128", "--num_encoder", "1", "--num_decoder", "1",
               "--rl_sample_number", "2", "--log_root", "/content/pergaze_gazeformer_smoke")


## Train or resume

The main train cell downloads COCO Mask R-CNN weights on first use, then uses
only its ResNet50 backbone. Backbone weights/BatchNorm are frozen and image
features are computed online inside each batch. Existing online checkpoints
restore backbone weights when resuming.

Every epoch evaluates test_seen, writes epoch metrics and last checkpoint,
and updates best.pth when the ScanMatch harmonic mean improves. supervised.pth
is saved after epoch 5. Progress bars show train, eval and subject retrieval.
Subprocess stdout/stderr are streamed; complete logs are copied to RUN/logs.
An error includes the last 40 output lines instead of only CalledProcessError.


In [ ]:
arguments = ["--log_root", str(RUN), "--workers", str(WORKERS),
             "--batch", str(TRAIN_BATCH), "--test_batch", str(TEST_BATCH),
             "--epoch", str(EPOCHS), "--start_rl_epoch", str(START_RL_EPOCH)]
if RESUME:
    arguments.append("--resume")
run_script("train.py", *arguments)


## Evaluate best.pth

This evaluates test_seen.json, which is the validation split. It is not an
independent unseen test. Metrics include ScanMatch, MultiMatch, SED/STDE and
subject retrieval, with full matrices and per-condition statistics in JSON.


In [ ]:
run_script("test.py", "--log_root", str(RUN), "--workers", str(WORKERS),
           "--test_batch", str(TEST_BATCH))


In [ ]:
import json
report_path = RUN / "evaluation_validation.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(json.dumps(report["overall"], indent=2))
print(json.dumps(report.get("retrieval", {}).get("by_condition", {}), indent=2))
print("Full report:", report_path)


## Resume and optional download

All checkpoints/config/reports are already on Drive. Keep RUN and TEXT_FILE.
After interruption, mount Drive, rerun setup with RESUME=True and unchanged
input/training settings, then run the train cell. An unfinished epoch restarts
from the last completed checkpoint.

For GazeformerISP-S, use RUN/checkpoints/best.pth and the exact TEXT_FILE.
The following cell prints both paths. Uncomment the ZIP lines to download a
copy of the run and text NPZ; this copies outputs only, not the dataset.


In [ ]:
print("Run saved on Drive:", RUN)
print("Text NPZ for resume/ISP-S:", TEXT_FILE)
# LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
# export_dir = LOCAL_ROOT / "export"
# shutil.copytree(RUN, export_dir / "pergaze_gazeformer_run", dirs_exist_ok=True)
# shutil.copy2(TEXT_FILE, export_dir / "gazeformer_task_embeddings.npz")
# archive = shutil.make_archive(str(LOCAL_ROOT / "pergaze_gazeformer_outputs"), "zip", root_dir=export_dir)
# from google.colab import files
# files.download(archive)
